In [1]:
import torch

# Paths
my_weights_path = "/auto/plzen4-ntis/home/ondrarot/SPUIA/UTMOSv2/models/spec_only/fold0_s42_best_model.pth"
patched_weights_path = "/auto/plzen4-ntis/home/ondrarot/SPUIA/UTMOSv2/models/spec_only/fold0_patched_for_inference.pth"

# Load your model
raw_data = torch.load(my_weights_path, map_location="cpu")

# Handle PyTorch Lightning's dictionary wrapper
if 'state_dict' in raw_data:
    state_dict = raw_data['state_dict']
else:
    state_dict = raw_data

# Find the exact name of the final scoring layer
fc_key = [k for k in state_dict.keys() if "fc.weight" in k][0]

# Create a new 5130-dimensional tensor filled with zeros
new_fc = torch.zeros((1, 5130), dtype=state_dict[fc_key].dtype, device=state_dict[fc_key].device)

# Copy your 5123 trained Czech weights into the first 5123 slots
new_fc[0, :5123] = state_dict[fc_key][0, :5123]

# Replace the old weights with the patched ones
state_dict[fc_key] = new_fc

# Save the updated dictionary back into the wrapper
if 'state_dict' in raw_data:
    raw_data['state_dict'] = state_dict
else:
    raw_data = state_dict
    
torch.save(raw_data, patched_weights_path)
print(f"Success! Patched weights saved to: {patched_weights_path}")

Success! Patched weights saved to: /auto/plzen4-ntis/home/ondrarot/SPUIA/UTMOSv2/models/spec_only/fold0_patched_for_inference.pth


In [1]:
# import utmosv2
# import pandas as pd
# import os
# import torch

# # 1. Setup paths
# data_dir = "/auto/plzen4-ntis/home/ondrarot/SPUIA/Mkunes_dataset/wavs"
# my_weights_path = "/auto/plzen4-ntis/home/ondrarot/SPUIA/UTMOSv2/models/spec_only/fold0_s42_best_model.pth"
# output_predictions_csv = "/auto/plzen4-ntis/home/ondrarot/SPUIA/my_model_predictions.csv"
# output_predictions_txt = "/auto/plzen4-ntis/home/ondrarot/SPUIA/my_model_predictions.txt"

# print("Loading the EXACT UTMOSv2 spec_only architecture...")
# # CRITICAL FIX: pretrained=False builds the 5123-dim model, exactly as you trained it!
# model = utmosv2.create_model(pretrained=False, device="cuda")

# print("Applying your CUSTOM model weights...")
# raw_state_dict = torch.load(my_weights_path, map_location="cuda")
# if 'state_dict' in raw_state_dict:
#     raw_state_dict = raw_state_dict['state_dict']

# clean_state_dict = {}
# for key, value in raw_state_dict.items():
#     # Just clean the names. NO PADDING HACK!
#     clean_key = key.replace("model.", "").replace("_model.", "")
#     clean_state_dict[clean_key] = value

# # Load your custom judge. strict=False tells PyTorch to ignore the missing SSL 'ear' weights 
# # because they were already loaded by create_model!
# model.load_state_dict(clean_state_dict, strict=False)
# model.eval()

# print(f"Evaluating the test files...")
# # Run prediction
# results = model.predict(input_dir=data_dir, device="cuda")

# # Format the output
# df = pd.DataFrame(results)
# df['file_path'] = df['file_path'].apply(os.path.basename)
# df.columns = ['file_name', 'predicted_mos']

# # Save BOTH versions!
# df.to_csv(output_predictions_csv, index=False, header=True)
# df.to_csv(output_predictions_txt, index=False, header=False)

# print(f"Success! Clean predictions saved to CSV (for you) and TXT (for the evaluation script).")

Loading the EXACT UTMOSv2 spec_only architecture...


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Applying your CUSTOM Czech weights...


RuntimeError: Error(s) in loading state_dict for SSLMultiSpecExtModelV2:
	size mismatch for fc.weight: copying a param with shape torch.Size([1, 5123]) from checkpoint, the shape in current model is torch.Size([1, 6686]).

In [ ]:
print('model created')